# Dev Notebook for hf-2.0 Branch

In [31]:
import json
import pandas as pd
import pyarrow as pa
import os

import torch

from transformers import AutoTokenizer, AutoImageProcessor

from PIL import Image

import sys
import os.path as osp
import json
import pickle
import time
import itertools
import skimage.io as io
# import matplotlib.pyplot as plt
# from matplotlib.collections import PatchCollection
# from matplotlib.patches import Polygon, Rectangle
from pprint import pprint
import numpy as np
# from refer import REFER

from tqdm import tqdm
from collections import defaultdict

# from renaissance.datasets.base_dataset import BaseDataset
# from renaissance.datamodules.datamodule_base import BaseDataModule
# from renaissance.datasets.refcoco_dataset import RefcocoDataset
# from renaissance.datamodules.refcoco_datamodule import RefcocoDataModule
from transformers import AutoModel, AutoTokenizer, AutoConfig

from renaissance.modules.renaissance_module import RenaissanceTransformer

In [37]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 1,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0,
        'cifar10' : 0
    }
    ret.update(d)
    return ret

config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["snli"],
    # 'loss_names' : _loss_names({"itm": 1, "mlm": 1}),
    'loss_names' : _loss_names({"snli": 1}),
    "batch_size" : 1,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "model_type" : 'one-tower',
    
    # One-Tower Settings
    "random_init_encoder" : False,
    "encoder" : "google-bert/bert-base-uncased",
    'encoder_type' : 'image',
    # Transformer Setting
    # 'vit' : "vit_base_patch32_384",
    'hidden_size' : 192,
    'num_heads' : 12,
    'num_layers' : 12,
    'mlp_ratio' : 4,
    'drop_rate' : 0.1,
    

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    'pooler_type' : 'double', 

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    # "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 2,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    'load_path' : '',
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}


In [38]:
model = RenaissanceTransformer(config)
model

RenaissanceTransformer(
  (encoder): OneTowerEncoder(
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (intermediate): BertIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act_fn): GELUActivation()
          )
          (output): BertOutp

In [39]:
hf_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")
hf_model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [40]:
hf_model.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[-0.0101, -0.0017,  0.0130,  ..., -0.0537,  0.0137, -0.0192],
        [-0.0604,  0.0104,  0.0648,  ..., -0.0486, -0.0277,  0.0070],
        [-0.0147,  0.0428,  0.0398,  ..., -0.0082,  0.0114, -0.0275],
        ...,
        [-0.0500, -0.0660,  0.0270,  ..., -0.0131,  0.0041,  0.0068],
        [ 0.0448, -0.0237,  0.0167,  ...,  0.0035,  0.0069,  0.0205],
        [-0.0094,  0.0363,  0.0258,  ..., -0.0086,  0.0064,  0.0604]],
       requires_grad=True)

In [41]:
model.encoder.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[-0.0101, -0.0017,  0.0130,  ..., -0.0537,  0.0137, -0.0192],
        [-0.0604,  0.0104,  0.0648,  ..., -0.0486, -0.0277,  0.0070],
        [-0.0147,  0.0428,  0.0398,  ..., -0.0082,  0.0114, -0.0275],
        ...,
        [-0.0500, -0.0660,  0.0270,  ..., -0.0131,  0.0041,  0.0068],
        [ 0.0448, -0.0237,  0.0167,  ...,  0.0035,  0.0069,  0.0205],
        [-0.0094,  0.0363,  0.0258,  ..., -0.0086,  0.0064,  0.0604]],
       requires_grad=True)

In [56]:
def _loss_names(d):
    ret = {
        "itm": 0,
        "mlm": 0,
        "mpp": 0,
        "vqa": 0,
        "vcr": 0,
        "vcr_qar": 0,
        "nlvr2": 0,
        "irtr": 0,
        "contras": 0,
        "snli": 0,
        "ref": 0,
        "mrpc" : 1,
        "rte" : 0,
        'wnli' : 0,
        'sst2' : 0,
        'qqp' : 0,
        'qnli' : 0,
        'mnli' : 0,
        'cola' : 0,
        'cifar10' : 0
    }
    ret.update(d)
    return ret

rand_config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["snli"],
    # 'loss_names' : _loss_names({"itm": 1, "mlm": 1}),
    'loss_names' : _loss_names({"snli": 1}),
    "batch_size" : 1,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.
    "model_type" : 'one-tower',
    
    # One-Tower Settings
    "random_init_encoder" : True,
    "encoder" : "google-bert/bert-base-uncased",
    'encoder_type' : 'image',
    # Transformer Setting
    # 'vit' : "vit_base_patch32_384",
    'encoder_manual_configuration' : False,
    'hidden_size' : 192,
    'num_heads' : 12,
    'num_layers' : 12,
    'mlp_ratio' : 4,
    'drop_rate' : 0.1,
    

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    'pooler_type' : 'double', 

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    # "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 2,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    'load_path' : '',
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}


In [57]:
rand_model = RenaissanceTransformer(rand_config)
rand_model

RenaissanceTransformer(
  (encoder): OneTowerEncoder(
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
          )
          (intermediate): BertIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act_fn): GELUActivation()
          )
          (output): BertOutp

In [58]:
rand_model.encoder.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[-0.0185,  0.0333,  0.0210,  ..., -0.0039, -0.0007,  0.0015],
        [ 0.0075,  0.0267,  0.0038,  ..., -0.0004, -0.0388,  0.0156],
        [-0.0086,  0.0017,  0.0007,  ...,  0.0010,  0.0014, -0.0120],
        ...,
        [-0.0196, -0.0118, -0.0181,  ..., -0.0348, -0.0213, -0.0454],
        [-0.0097,  0.0060,  0.0122,  ..., -0.0137,  0.0224,  0.0133],
        [-0.0078,  0.0148, -0.0201,  ..., -0.0076, -0.0334,  0.0350]],
       requires_grad=True)

In [59]:
rand_config = AutoConfig.from_pretrained('google-bert/bert-base-uncased')
rand_hf_model = AutoModel.from_config(rand_config)
rand_hf_model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [60]:
rand_hf_model.encoder.layer[0].intermediate.dense.weight

Parameter containing:
tensor([[-1.9937e-02,  1.2120e-02, -4.2187e-03,  ..., -1.0200e-02,
          2.2540e-02, -1.5180e-02],
        [ 4.1884e-03,  8.9673e-03, -1.2876e-03,  ..., -1.3967e-02,
         -1.3269e-02,  4.9154e-04],
        [ 5.1158e-03,  1.4830e-02,  8.0921e-03,  ...,  1.6861e-02,
         -1.0104e-02,  1.0561e-02],
        ...,
        [-4.3636e-02,  2.6136e-03,  2.0225e-02,  ..., -2.2246e-02,
          1.0059e-03,  1.0462e-03],
        [-1.9618e-03,  9.0589e-03, -3.9262e-05,  ...,  1.4003e-02,
          4.8808e-03,  6.5562e-03],
        [ 2.3893e-02, -8.9497e-04, -1.5895e-02,  ..., -4.2346e-03,
          7.6938e-03,  9.6727e-03]], requires_grad=True)

## DataModule

In [3]:
dm = MTDataModule(config, dist=False)

In [4]:
model = METERTransformerSS(config)

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
model

METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (cross_modal_image_layers): ModuleList(
    (0-5): 6 x BertCrossLayer(
      (attention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): BertSelfOutput(
          (dense): Linear(in_features=256, out_features=256, bias=True)
          (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (crossattention): BertAttention(
        (self): BertSelfAttention(
          (query): Linear(in_features=256, out_features=256, bias=

In [6]:
dm.prepare_data()
dm.setup('train')

In [7]:
dl = dm.train_dataloader()

In [8]:
batch = next(iter(dl))

You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenize

In [9]:
batch

{'labels': [0, 2],
 'image': [tensor([[[[ 1.1529,  0.7248,  0.6906,  ...,  0.5193,  0.2111,  0.7248],
            [ 0.4166, -0.0116,  1.1872,  ...,  0.4508,  0.2624,  0.6392],
            [-0.5424, -0.7479, -0.3369,  ...,  0.5536,  0.2624,  0.7591],
            ...,
            [ 0.5022,  0.5364,  0.5364,  ..., -1.1932, -1.1418, -1.1589],
            [ 0.7762,  0.7591,  0.7419,  ..., -0.8849, -0.8678, -0.9192],
            [ 0.9474,  0.8789,  0.8618,  ..., -0.0801,  0.0569,  0.0056]],
  
           [[ 0.9930,  0.6604,  0.7654,  ...,  0.8354,  0.4678,  0.9230],
            [ 0.5028,  0.1527,  1.2731,  ...,  0.7304,  0.5203,  0.8880],
            [-0.5651, -0.5301, -0.1625,  ...,  0.8354,  0.5028,  1.0280],
            ...,
            [ 0.5903,  0.6254,  0.6254,  ..., -1.1429, -1.1078, -1.0903],
            [ 0.8704,  0.8529,  0.8354,  ..., -0.7052, -0.7052, -0.7752],
            [ 1.0455,  0.9755,  0.9580,  ...,  0.1176,  0.2052,  0.2052]],
  
           [[ 0.6008,  0.4962,  1.0365,  .

In [10]:
text_encoder = model.text_transformer
image_encoder = model.image_encoder
cross_modal_image_transform = model.cross_modal_image_transform
cross_modal_text_transform = model.cross_modal_text_transform
cross_modal_text_transform

Linear(in_features=256, out_features=256, bias=True)

In [11]:
text_hidden_state = text_encoder( input_ids = batch['text_ids'], attention_mask = batch['text_masks'])[0]
text_hidden_state = cross_modal_text_transform(text_hidden_state)
text_hidden_state.shape

torch.Size([2, 128, 256])

In [12]:
image_hidden_state = image_encoder(batch['image'][0])[0]
image_hidden_state = cross_modal_image_transform(image_hidden_state)
image_hidden_state.shape

torch.Size([2, 197, 256])

In [13]:
# model.text_transformer.get_extended_attention_mask(image_masks, image_masks.size())

In [14]:
# model.fusion_encoder.cross_modal_image_layers[0]

## Use Huggingface for CrossLayers

In [15]:
from meter.modules.bert_model import BertCrossLayer
from transformers.models.bert.modeling_bert import BertLayer
from transformers.models.bert.configuration_bert import BertConfig

import torch.nn as nn

In [16]:
bert_config = BertConfig(
            vocab_size=config["vocab_size"],
            hidden_size=config["cross_layer_hidden_size"],
            num_attention_heads=config["num_cross_layer_heads"],
            intermediate_size=config["cross_layer_hidden_size"] * config["cross_layer_mlp_ratio"],
            max_position_embeddings=config["max_text_len"],
            hidden_dropout_prob=config["cross_layer_drop_rate"],
            attention_probs_dropout_prob=config["cross_layer_drop_rate"],
)
bert_config

BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 1024,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 128,
  "model_type": "bert",
  "num_attention_heads": 4,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.36.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [17]:
bert_config = BertConfig(config, add_cross_attention=True, is_decoder=True, 
                         hidden_size=config['cross_layer_hidden_size'],
                        num_attention_heads=4)
bert_config

BertConfig {
  "add_cross_attention": true,
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 256,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": true,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 4,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.36.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": {
    "batch_size": 1,
    "cross_layer_drop_rate": 0.1,
    "cross_layer_hidden_size": 256,
    "cross_layer_mlp_ratio": 4,
    "data_root": "/home/claytonfields/nlp/code/meter/data/arrow",
    "datasets": [
      "snli"
    ],
    "decay_power": 1,
    "draw_false_image": 1,
    "draw_false_text": 0,
    "drop_rate": 0.1,
    "encoder": "facebook/deit-tiny-patch16-224",
    "encoder_type": "image",
    "end_lr": 0,
    "exp_na

### BertLayer from models/bert_modeling.py

In [18]:
cross_modal_text_layers_bert = nn.ModuleList([BertLayer(bert_config) for _ in range(config['num_cross_layers'])])
cross_modal_image_layers_bert = nn.ModuleList([BertLayer(bert_config) for _ in range(config['num_cross_layers'])])

In [19]:
cross_modal_image_layers_bert

ModuleList(
  (0-5): 6 x BertLayer(
    (attention): BertAttention(
      (self): BertSelfAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (output): BertSelfOutput(
        (dense): Linear(in_features=256, out_features=256, bias=True)
        (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (crossattention): BertAttention(
      (self): BertSelfAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (output): BertSelfOutput(
        (dense): Linear(in_

In [20]:
do_mlm = ''
text_ids = batch[f"text_ids{do_mlm}"]
text_labels = batch[f"text_labels{do_mlm}"]
text_masks = batch["text_masks"]

In [21]:
text_masks.shape

torch.Size([2, 128])

In [22]:
input_shape = text_masks.size()
extend_text_masks = text_encoder.get_extended_attention_mask(text_masks, input_shape)
extend_text_masks.shape

torch.Size([2, 1, 1, 128])

In [23]:
image_embeds = image_encoder(batch['image'][0])[0]
image_embeds.shape

torch.Size([2, 197, 192])

In [24]:
image_hidden_state.shape

torch.Size([2, 197, 256])

In [25]:
image_masks = torch.ones((image_hidden_state.size(0), image_hidden_state.size(1)), dtype=torch.long)
extend_image_masks = text_encoder.get_extended_attention_mask(image_masks, image_masks.size())
extend_image_masks.shape

torch.Size([2, 1, 1, 197])

In [26]:
x, y = text_hidden_state, image_hidden_state
for text_layer, image_layer in zip(cross_modal_text_layers_bert, cross_modal_image_layers_bert):
    # x1 = text_layer(hidden_states=x, encoder_hidden_states=y, attention_mask=extend_text_masks, encoder_attention_mask=extend_image_masks)
    # y1 = image_layer(hidden_states=y, encoder_hidden_states=x, attention_mask=extend_image_masks, encoder_attention_mask=extend_text_masks)
    x1 = text_layer(hidden_states=x, encoder_hidden_states=y)
    y1 = image_layer(hidden_states=y, encoder_hidden_states=x)
    x, y = x1[0], y1[0]

text_feats, image_feats = x, y
# cls_feats_text = self.cross_modal_text_pooler(x)
# cls_feats_image = self.cross_modal_image_pooler(y)
# cls_feats = torch.cat([cls_feats_text, cls_feats_image], dim=-1)
print(text_feats)
image_feats

tensor([[[-0.2953,  0.6192, -0.7921,  ...,  0.5858, -0.5391,  0.4198],
         [-0.1289, -0.2816, -0.1271,  ..., -0.3247, -0.1244, -0.5523],
         [ 0.7240,  1.0134,  0.0735,  ..., -1.9968, -0.4397, -1.6123],
         ...,
         [ 0.8813, -0.9632, -0.2872,  ..., -1.0258,  0.1817,  0.4515],
         [ 0.6850, -0.1311, -0.6777,  ..., -1.0895, -0.0517,  0.5668],
         [ 0.5216, -1.4151, -0.2270,  ..., -0.4445,  0.1073,  0.7534]],

        [[-0.2559, -0.5505, -0.6499,  ...,  0.3038, -0.4781,  0.9161],
         [ 0.1343, -0.0521,  0.0872,  ..., -0.2974, -0.0239, -0.6960],
         [ 0.8837, -0.1909, -0.0313,  ..., -0.5204,  0.1001, -0.3911],
         ...,
         [ 0.4884, -0.8502, -0.4352,  ..., -1.1608,  0.0861,  0.3439],
         [ 0.8418, -1.1572, -0.6677,  ..., -0.5960,  0.2097,  0.5036],
         [ 0.9897, -0.9303, -0.1252,  ..., -1.2270,  0.3702,  0.4600]]],
       grad_fn=<NativeLayerNormBackward0>)


tensor([[[ 0.6243,  1.1633, -0.2898,  ...,  0.0601,  0.9471,  0.6720],
         [-0.0883,  1.0271, -0.5672,  ..., -2.2876,  0.9955, -0.1500],
         [ 1.3873,  0.7217,  0.4683,  ..., -0.9206,  0.5505,  0.3144],
         ...,
         [ 1.5681,  1.0598, -0.6162,  ..., -1.7572,  0.3193, -1.0429],
         [ 1.0030,  0.7861,  1.2704,  ..., -1.4063,  1.3730, -0.4802],
         [ 0.7793, -0.5463,  0.9608,  ..., -0.8807,  0.8908,  0.7428]],

        [[ 0.8054,  1.6410,  0.7094,  ..., -0.4101,  0.6355,  1.7091],
         [ 0.4242,  1.0237, -0.4046,  ..., -1.6230,  1.2286, -0.3931],
         [ 0.9448,  0.7549, -0.2207,  ..., -0.7263,  0.4088,  0.3576],
         ...,
         [ 1.8510,  0.7109, -0.3439,  ..., -1.8955,  0.7312, -1.3713],
         [ 1.4147, -0.0731,  0.5323,  ..., -1.9521,  1.0640,  0.6269],
         [ 1.2927, -0.1449,  1.4817,  ..., -0.4841,  0.8132,  1.3994]]],
       grad_fn=<NativeLayerNormBackward0>)

Why the extended masks?

What are they?

**Are they necessary?**

They appear to be necessary.

### LxmertCrossModalEncoder

In [27]:
from transformers.models.lxmert.modeling_lxmert import LxmertXLayer

In [28]:
cross_modal_layers_lx = nn.ModuleList([LxmertXLayer(bert_config) for _ in range(config['num_cross_layers'])])
# cross_modal_image_layers_lx = nn.ModuleList([LxmertXLayer(bert_config) for _ in range(config['num_cross_layers'])])

In [29]:
# x, y = text_hidden_state, image_hidden_state
# for text_layer, image_layer in zip(cross_modal_text_layers_lx, cross_modal_image_layers_lx):
#     # x1 = text_layer(hidden_states=x, encoder_hidden_states=y, attention_mask=extend_text_masks, encoder_attention_mask=extend_image_masks)
#     # y1 = image_layer(hidden_states=y, encoder_hidden_states=x, attention_mask=extend_image_masks, encoder_attention_mask=extend_text_masks)
#     x1 = text_layer(x, extend_text_masks, y, extend_image_masks)
#     y1 = image_layer(y, extend_image_masks,  x, extend_text_masks)
#     x, y = x1[0], y1[0]

# text_feats, image_feats = x, y
# # cls_feats_text = self.cross_modal_text_pooler(x)
# # cls_feats_image = self.cross_modal_image_pooler(y)
# # cls_feats = torch.cat([cls_feats_text, cls_feats_image], dim=-1)
# print(text_feats)
# image_feats

In [30]:
print(x.shape)
print(y.shape)

torch.Size([2, 128, 256])
torch.Size([2, 197, 256])


In [31]:
vision_hidden_states = ()
language_hidden_states = ()

In [32]:
for layer_module in cross_modal_layers_lx:
    
    x_outputs = layer_module(
        x, 
        extend_text_masks, 
        y, 
        extend_image_masks
    )
    lang_feats, visual_feats = x_outputs[:2]
#     vision_hidden_states = vision_hidden_states + (visual_feats,)
#     language_hidden_states = language_hidden_states + (lang_feats,)
    
# visual_encoder_outputs = (
#     vision_hidden_states,
#     vision_attentions if output_attentions else None,
# )
# lang_encoder_outputs = (
#     language_hidden_states,
#     language_attentions if output_attentions else None,
# )
print(lang_feats.shape)
visual_feats.shape

torch.Size([2, 128, 256])


torch.Size([2, 197, 256])

In [33]:
len(vision_hidden_states + (visual_feats,))

1

### TODO: Test both cross modal encoders!

In [34]:
from meter.modules.fusion_encoder import BertCrossModalEncoder, LxmertCrossModalEncoder

In [35]:
lx_encoder = LxmertCrossModalEncoder(config)

In [36]:
lx_encoder

LxmertCrossModalEncoder(
  (cross_modal_layers): ModuleList(
    (0-5): 6 x LxmertXLayer(
      (visual_attention): LxmertCrossAttentionLayer(
        (att): LxmertAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (output): LxmertAttentionOutput(
          (dense): Linear(in_features=256, out_features=256, bias=True)
          (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (lang_self_att): LxmertSelfAttentionLayer(
        (self): LxmertAttention(
          (query): Linear(in_features=256, out_features=256, bias=True)
          (key): Linear(in_features=256, out_features=256, bias=True)
          (value): Linear(in_features=256, out_features=256, bias=T

In [37]:
output = lx_encoder(text_hidden_state,extend_text_masks,image_hidden_state,extend_image_masks)

In [38]:
output[0]

tensor([[ 8.4365e-01, -7.2157e-01, -2.4035e-04,  ...,  3.4894e-01,
         -6.5354e-01, -4.0124e-01],
        [ 7.0431e-01, -7.2051e-01, -3.4538e-01,  ...,  1.8013e-01,
         -2.9707e-01, -2.6959e-01]], grad_fn=<CatBackward0>)

In [39]:
image_hidden_state.shape

torch.Size([2, 197, 256])

In [49]:
output[0].shape

torch.Size([2, 512])

In [40]:
from meter.modules.fusion_encoder import BertCrossModalEncoder

In [41]:
br_encoder = BertCrossModalEncoder(config)

In [42]:
br_output = br_encoder(text_hidden_state,image_hidden_state,extend_text_masks,extend_image_masks)

In [50]:
br_output[0].shape

torch.Size([2, 512])

In [44]:
text_hidden_state

tensor([[[-2.6552e-02,  1.0543e-01, -1.1289e-01,  ...,  6.4151e-01,
          -2.3747e-01,  1.8389e-01],
         [-4.5087e-02,  3.5396e-01,  7.3572e-02,  ...,  2.5380e-01,
          -1.7905e-01, -3.1934e-01],
         [ 1.1626e-01,  4.6027e-01,  1.3728e-01,  ..., -3.7711e-01,
          -1.8092e-01, -4.1459e-01],
         ...,
         [-4.0443e-02, -1.5931e-01, -5.0290e-02,  ..., -1.4145e-01,
           3.2540e-02,  1.5132e-01],
         [-5.7957e-02, -1.8133e-01, -5.1774e-02,  ..., -1.2786e-01,
           2.4721e-02,  1.5291e-01],
         [-6.1887e-02, -1.9323e-01, -4.6489e-02,  ..., -1.2412e-01,
           3.4015e-02,  1.4902e-01]],

        [[-1.3171e-01,  8.5139e-02, -1.2550e-02,  ...,  4.8763e-01,
          -2.2894e-01,  6.0261e-02],
         [ 2.6440e-02,  3.0343e-01,  6.0485e-02,  ...,  1.2859e-01,
          -1.8364e-01, -3.0289e-01],
         [ 2.4829e-01,  3.8817e-01,  2.8448e-02,  ...,  3.8942e-02,
          -2.2088e-02, -5.7234e-02],
         ...,
         [-7.1338e-03, -1

In [45]:
batch

{'labels': [0, 2],
 'image': [tensor([[[[ 1.1529,  0.7248,  0.6906,  ...,  0.5193,  0.2111,  0.7248],
            [ 0.4166, -0.0116,  1.1872,  ...,  0.4508,  0.2624,  0.6392],
            [-0.5424, -0.7479, -0.3369,  ...,  0.5536,  0.2624,  0.7591],
            ...,
            [ 0.5022,  0.5364,  0.5364,  ..., -1.1932, -1.1418, -1.1589],
            [ 0.7762,  0.7591,  0.7419,  ..., -0.8849, -0.8678, -0.9192],
            [ 0.9474,  0.8789,  0.8618,  ..., -0.0801,  0.0569,  0.0056]],
  
           [[ 0.9930,  0.6604,  0.7654,  ...,  0.8354,  0.4678,  0.9230],
            [ 0.5028,  0.1527,  1.2731,  ...,  0.7304,  0.5203,  0.8880],
            [-0.5651, -0.5301, -0.1625,  ...,  0.8354,  0.5028,  1.0280],
            ...,
            [ 0.5903,  0.6254,  0.6254,  ..., -1.1429, -1.1078, -1.0903],
            [ 0.8704,  0.8529,  0.8354,  ..., -0.7052, -0.7052, -0.7752],
            [ 1.0455,  0.9755,  0.9580,  ...,  0.1176,  0.2052,  0.2052]],
  
           [[ 0.6008,  0.4962,  1.0365,  .

In [51]:
model.infer(batch)['cls_feats'].shape

torch.Size([2, 512])